# FinGuard AI — Model Evaluation Notebook

This notebook uses the bundled dummy transaction statement. It demonstrates cleaning, EDA, regression comparison, classification comparison, risk modelling, anomaly detection, and the five bundled Joblib artifacts. Metrics from synthetic/demo data are educational, not production benchmarks.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.data_processing import clean_transactions_dataframe
from backend.ml_service import (
    compare_category_classifiers,
    detect_anomalies,
    forecast_monthly_expense,
    load_model_registry,
    predict_monthly_savings,
    train_financial_risk_classifier,
)
from backend.eda import category_boxplot_figure, correlation_heatmap_figure, monthly_expense_figure
print(ROOT)


## 1. Load and clean the sample bank statement


In [ ]:
raw = pd.read_csv(ROOT / 'datasets' / 'sample_transactions.csv')
cleaned = clean_transactions_dataframe(raw)
cleaned.head(), cleaned.shape


## 2. Prepare the analytics dataframe


In [ ]:
df = cleaned.rename(columns={
    'transaction_date': 'date',
    'transaction_type': 'type',
}).copy()
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').astype(str)
df['weekday'] = df['date'].dt.day_name()
df['risk_level'] = df.get('risk_level', 'LOW')
df['is_unusual'] = df.get('is_unusual', False)
df.head()


## 3. Matplotlib and Seaborn EDA


In [ ]:
monthly_expense_figure(df)
plt.show()
category_boxplot_figure(df)
plt.show()
correlation_heatmap_figure(df)
plt.show()


## 4. Regression and savings prediction


In [ ]:
expense_result = forecast_monthly_expense(df, user_id=9999)
savings_result = predict_monthly_savings(df, user_id=9999)
pd.DataFrame(expense_result['metrics']), savings_result


## 5. Expense category classification


In [ ]:
category_result = compare_category_classifiers(df, user_id=9999)
pd.DataFrame(category_result['metrics']), category_result['best_model'], category_result['confusion_matrix']


## 6. Financial risk classification and anomaly detection


In [ ]:
risk_result = train_financial_risk_classifier(df, user_id=9999)
anomaly_df = detect_anomalies(df, user_id=9999)
pd.DataFrame(risk_result['metrics']), anomaly_df[anomaly_df['ml_anomaly']].head()


## 7. Validate bundled model registry


In [ ]:
registry = pd.DataFrame(load_model_registry())
registry


## Interpretation

Choose the best classifier using weighted F1, and the best regressor using the lowest RMSE/MAE. Confusion matrices show class-level mistakes. Isolation Forest flags unusual amounts but does not prove fraud. Financial risk labels in this academic build are rule-assisted and should not be presented as bank-certified fraud labels.
